# Linguistic Forensic Audit: NER & POS Drift

**Ship of Theseus — Computational Forensics**  
CS 6120 · Spring 2026 · Northeastern University

This notebook measures how POS-tag distributions (style) and named entities (content) erode across paraphrasing iterations, directly informing **RQ1: Style vs. Content Decay**.

---

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import (
    ALL_DATASETS, VERSION_TO_COL, DATA_PROCESSED,
    SPACY_MODEL, FIGURES_DIR, EXPERIMENTS_DIR,
)
from src.data.load_data import load_paired_t1

TEXT_COLS = list(VERSION_TO_COL.values())

print(f"Project root : {PROJECT_ROOT}")
print(f"Datasets     : {ALL_DATASETS}")

---
## Task 1 — Data Processing & Version Alignment

**Goal:** Pair the original (T0) text of each article with its T1 paraphrased versions across multiple paraphrasers, using `(key, source, dataset)` as the composite identifier.

| Output column | Version string | Paraphraser |
|---|---|---|
| `text_T0` | `original` | — |
| `text_chatgpt` | `chatgpt` | ChatGPT |
| `text_dipper_high` | `dipper(high)` | Dipper (High) |
| `text_dipper_low` | `dipper(low)` | Dipper (Low) |
| `text_pegasus_slight` | `pegasus(slight)` | Pegasus (Slight) |
| `text_pegasus_full` | `pegasus(full)` | Pegasus (Full) |

Version parsing uses a regex that finds the shortest repeating base token — no naive underscore splitting.  
The `orignal` typo is normalised to `original`. See `src/utils/config.parse_version` for details.

In [ ]:
paired = load_paired_t1(cache=True)

### 1c. Deliverable

Print `paired.shape`, display 2 rows per dataset (14 total), and confirm the `dataset` column and `text_dipper_low` column are present.

In [ ]:
print(f"paired.shape = {paired.shape}")
print(f"\nColumns: {list(paired.columns)}")
print(f"\n'dataset' column present : {'dataset' in paired.columns}")
print(f"'text_dipper_low' present: {'text_dipper_low' in paired.columns}")

# 2 rows per dataset
sample = paired.groupby("dataset").head(2)
print(f"\nSample: 2 rows per dataset ({len(sample)} rows total)")

display_cols = ["key", "source", "dataset"] + TEXT_COLS
display_df = sample[display_cols].copy()
for col in TEXT_COLS:
    display_df[col] = display_df[col].apply(
        lambda x: x[:80] + "..." if isinstance(x, str) and len(x) > 80 else x
    )
display_df

In [ ]:
print("Articles per dataset:")
print(paired.groupby("dataset").size().to_string())
print(f"\nTotal articles: {len(paired):,}")
print(f"Unique sources: {sorted(paired['source'].unique())}")

---

*To reload on subsequent runs, `load_paired_t1(cache=True)` reads from pickle automatically.*